# 第9天：机器学习数据准备与基线认识

本节先通过少量用户理解“特征、标签和预测”，再把清洗后的电商用户数据整理成可供分类模型使用的数值矩阵。

> 机器学习会从带有真实结果的历史用户中寻找规律，并把这些规律用于未参与训练的用户。

## 任务0：环境和个人信息

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "data/ecommerce_customer_cleaned.csv"
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)
RANDOM_STATE = 42
TEST_SIZE = 0.20
pd.set_option("display.max_columns", 80)

In [2]:
STUDENT_NAME = "24012412"
STUDENT_ID = "24012412"
CLASS_NAME = "信计二班"
assert STUDENT_NAME.strip() and STUDENT_ID == "24012412" and CLASS_NAME.strip(), "个人信息不能为空"

## 任务1：用6名用户区分规则、特征与标签

人工规则设为：使用月数不超过3个月且发生过投诉时，预测该用户会流失。`Tenure`和`Complain`是判断线索，`Churn`是已有的真实结果。

In [3]:
toy = pd.DataFrame({
    "用户": ["甲", "乙", "丙", "丁", "戊", "己"],
    "Tenure": [2, 20, 4, 16, 1, 28],
    "Complain": [1, 0, 0, 1, 1, 0],
    "Churn": [1, 0, 1, 0, 1, 0],
})
rule_hit = (toy["Tenure"] <= 3) & (toy["Complain"] == 1)
toy["人工规则预测"] = rule_hit.astype(int)
toy["是否判断正确"] = toy["人工规则预测"].eq(toy["Churn"])
display(toy)
print("判断正确：", int(toy["是否判断正确"].sum()), "/", len(toy))
print("真实流失3人，规则识别：", int(((toy["人工规则预测"] == 1) & (toy["Churn"] == 1)).sum()), "人")

,用户,Tenure,Complain,Churn,人工规则预测,是否判断正确
0,甲,2,1,1,1,True
1,乙,20,0,0,0,True
2,丙,4,0,1,0,False
3,丁,16,1,0,0,True
4,戊,1,1,1,1,True
5,己,28,0,0,0,True


判断正确： 5 / 6
真实流失3人，规则识别： 2 人


**思考：** 丙用户虽然真实流失，但没有同时满足“3个月以内”和“投诉”两个条件，因此被规则漏掉。人工规则由人预先设定；机器学习则利用大量样本，综合多个特征学习判断规律。

## 任务2：读取真实数据并认识样本与标签

In [4]:
df = pd.read_csv(DATA_PATH)
print("数据形状：", df.shape)
print("一行代表一名用户；总体流失率：", f"{df['Churn'].mean():.2%}")
assert df.shape == (5630, 22)
assert df["CustomerID"].is_unique
assert set(df["Churn"].unique()) == {0, 1}
assert int(df.isna().sum().sum()) == 0

数据形状： (5630, 22)
一行代表一名用户；总体流失率： 16.84%


## 任务3：确定建模口径

每一行是一名用户。`X`只保留可用于判断的特征，`y`单独保存需要预测的流失标签；用户编号只用于识别和回查。

In [5]:
TARGET = "Churn"
ID_COL = "CustomerID"
feature_columns = [c for c in df.columns if c not in {TARGET, ID_COL}]
X = df.loc[:, feature_columns].copy()
y = df[TARGET].astype("int64").copy()
assert TARGET not in X.columns and ID_COL not in X.columns
assert len(X) == len(y) == 5630
print("特征表：", X.shape, "标签：", y.shape)

特征表： (5630, 20) 标签： (5630,)


## 任务4：查看特征方案

代码自动区分数值列、文字类别和派生特征。今天重点理解处理目的，不要求默写。

In [6]:
categorical_features = X.select_dtypes(include=["object", "string"]).columns.tolist()
numeric_features = [column for column in X.columns if column not in categorical_features]
derived_features = {"TenureGroup", "IsMobileLogin"}

schema_rows = []
for column in df.columns:
    if column == ID_COL:
        item = ("identifier", "drop", "编号只用于定位用户，不参与规律学习")
    elif column == TARGET:
        item = ("target", "separate", "流失结果需要从特征中单独拿出")
    elif column in derived_features:
        item = ("derived_feature", "candidate", "由原始字段转换得到，可作为补充线索")
    elif column in categorical_features:
        item = ("categorical_feature", "one_hot", "文字类别需转换为模型可读取的数值列")
    else:
        item = ("numeric_feature", "numeric_pipeline", "数值特征进入填补与统一尺度步骤")
    role, action, reason = item
    schema_rows.append({
        "feature": column,
        "role": role,
        "dtype": str(df[column].dtype),
        "action": action,
        "reason": reason,
    })

feature_schema = pd.DataFrame(schema_rows)
feature_schema.to_csv(OUTPUT_DIR / "feature_schema.csv", index=False, encoding="utf-8-sig")
display(feature_schema)

,feature,role,dtype,action,reason
0,CustomerID,identifier,int64,drop,编号只用于定位用户，不参与规律学习
1,Churn,target,int64,separate,流失结果需要从特征中单独拿出
2,Tenure,numeric_feature,float64,numeric_pipeline,数值特征进入填补与统一尺度步骤
3,PreferredLoginDevice,categorical_feature,object,one_hot,文字类别需转换为模型可读取的数值列
4,CityTier,numeric_feature,int64,numeric_pipeline,数值特征进入填补与统一尺度步骤
5,WarehouseToHome,numeric_feature,float64,numeric_pipeline,数值特征进入填补与统一尺度步骤
6,PreferredPaymentMode,categorical_feature,object,one_hot,文字类别需转换为模型可读取的数值列
7,Gender,categorical_feature,object,one_hot,文字类别需转换为模型可读取的数值列
8,HourSpendOnApp,numeric_feature,float64,numeric_pipeline,数值特征进入填补与统一尺度步骤
9,NumberOfDeviceRegistered,numeric_feature,int64,numeric_pipeline,数值特征进入填补与统一尺度步骤


## 任务5：把数据分成训练集和测试集

训练集用于学习，测试集模拟没有见过的新用户。

In [7]:
STRATIFY_TARGET = y
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=STRATIFY_TARGET,
)
assert STRATIFY_TARGET is y, "分层依据必须使用标签y"

In [8]:
split_records = []
for split_name, split_x, split_y in [
    ("train", X_train, y_train),
    ("test", X_test, y_test),
]:
    split_records.append({
        "split": split_name,
        "rows": len(split_x),
        "churn_count": int(split_y.sum()),
        "churn_rate": float(split_y.mean()),
    })
split_summary = pd.DataFrame(split_records)
split_summary.to_csv(OUTPUT_DIR / "split_summary.csv", index=False, encoding="utf-8-sig")
display(split_summary)
assert split_summary["rows"].sum() == 5630
assert split_summary["churn_rate"].max() - split_summary["churn_rate"].min() < 0.01

,split,rows,churn_count,churn_rate
0,train,4504,758,0.168295
1,test,1126,190,0.168739


## 任务6：运行教师提供的预处理流水线

- 数值分支：缺失兜底并统一尺度；
- 类别分支：把文字类别转为0/1列；
- 流水线只能从训练集学习规则。

In [9]:
numeric_pipeline = Pipeline([
    ("fill_median", SimpleImputer(strategy="median")),
    ("standardize", StandardScaler()),
])
categorical_pipeline = Pipeline([
    ("fill_mode", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features),
])

# 所有规则只在训练集上拟合；测试集只调用已学好的转换规则。
X_train_ready = preprocessor.fit_transform(X_train)
X_test_ready = preprocessor.transform(X_test)
feature_names = preprocessor.get_feature_names_out()

assert X_train_ready.shape == (4504, 36)
assert X_test_ready.shape == (1126, 36)
assert np.isfinite(X_train_ready).all() and np.isfinite(X_test_ready).all()
model_matrix_preview = pd.DataFrame(X_train_ready[:20], columns=feature_names)
model_matrix_preview.to_csv(OUTPUT_DIR / "model_matrix_preview.csv", index=False, encoding="utf-8-sig")
print("训练矩阵：", X_train_ready.shape, "测试矩阵：", X_test_ready.shape)
display(model_matrix_preview.head())

训练矩阵： (4504, 36) 测试矩阵： (1126, 36)


,num__Tenure,num__CityTier,num__WarehouseToHome,num__HourSpendOnApp,num__NumberOfDeviceRegistered,num__SatisfactionScore,num__NumberOfAddress,num__Complain,num__OrderAmountHikeFromlastYear,num__CouponUsed,num__OrderCount,num__DaySinceLastOrder,num__CashbackAmount,num__IsMobileLogin,cat__PreferredLoginDevice_Computer,cat__PreferredLoginDevice_Mobile Phone,cat__PreferredPaymentMode_Cash on Delivery,cat__PreferredPaymentMode_Credit Card,cat__PreferredPaymentMode_Debit Card,cat__PreferredPaymentMode_E wallet,cat__PreferredPaymentMode_UPI,cat__Gender_Female,cat__Gender_Male,cat__PreferedOrderCat_Fashion,cat__PreferedOrderCat_Grocery,cat__PreferedOrderCat_Laptop & Accessory,cat__PreferedOrderCat_Mobile Phone,cat__PreferedOrderCat_Others,cat__MaritalStatus_Divorced,cat__MaritalStatus_Married,cat__MaritalStatus_Single,cat__TenureGroup_0-6个月,cat__TenureGroup_13-24个月,cat__TenureGroup_25-36个月,cat__TenureGroup_36个月以上,cat__TenureGroup_7-12个月
0,-0.135398,1.453559,0.043177,-1.325711,-0.675976,-1.493573,-0.856755,-0.623912,-0.468858,3.386635,2.157964,0.715754,0.456593,-1.563146,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
1,-0.494759,1.453559,-0.318966,-2.742656,-0.675976,0.673881,-1.246653,-0.623912,0.366481,-0.926500,-0.683571,-0.691017,-0.700031,-1.563146,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,-0.255185,-0.724000,-0.077537,0.091233,0.294877,0.673881,2.262432,-0.623912,0.923374,-0.926500,-0.683571,-1.253725,-0.244144,0.639736,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
3,0.583322,1.453559,-0.560394,0.091233,-0.675976,0.673881,2.262432,1.602791,0.923374,2.847493,1.447580,0.715754,0.144113,0.639736,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
4,0.223962,-0.724000,-0.318966,1.508178,1.265730,-0.048604,-0.076958,-0.623912,-1.025752,0.690925,1.802772,0.997108,0.016799,-1.563146,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0


## 任务7：运行最低参照线

它永远预测人数最多的类别，只用来回答：正式模型至少要超过什么水平？

In [10]:
baseline = DummyClassifier(strategy="prior", random_state=RANDOM_STATE)
baseline.fit(X_train_ready, y_train)
y_pred = baseline.predict(X_test_ready)

metric_values = {
    "accuracy": accuracy_score(y_test, y_pred),
    "churn_recall": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
    "predicted_churn_count": int(y_pred.sum()),
}
baseline_metrics = pd.DataFrame([
    {"metric": name, "value": value} for name, value in metric_values.items()
])
baseline_metrics.to_csv(OUTPUT_DIR / "baseline_metrics.csv", index=False, encoding="utf-8-sig")
display(baseline_metrics)
print("测试集真实流失人数：", int(y_test.sum()))
print("最低参照线预测流失人数：", int(y_pred.sum()))

,metric,value
0,accuracy,0.831261
1,churn_recall,0.000000
2,predicted_churn_count,0.000000


测试集真实流失人数： 190
最低参照线预测流失人数： 0


## 任务8：写出自己的解释

In [11]:
reflection = "一行数据就是一名用户，特征是模型判断时能看到的线索，标签Churn是要预测的答案。CustomerID只负责区分用户，不能当作特征。训练集用于学习处理规则和分类规律，测试集必须保留到最后，模拟新用户。基线准确率约83%，但它把所有人都判断为不流失，流失召回率为0，说明高准确率并不等于能找到真正需要挽留的用户。"
assert 100 <= len(reflection) <= 200, "复盘长度应为100～200字"
print(reflection)

一行数据就是一名用户，特征是模型判断时能看到的线索，标签Churn是要预测的答案。CustomerID只负责区分用户，不能当作特征。训练集用于学习处理规则和分类规律，测试集必须保留到最后，模拟新用户。基线准确率约83%，但它把所有人都判断为不流失，流失召回率为0，说明高准确率并不等于能找到真正需要挽留的用户。


## 提交检查

In [12]:
required = {"feature_schema.csv", "split_summary.csv", "model_matrix_preview.csv", "baseline_metrics.csv"}
actual = {path.name for path in OUTPUT_DIR.glob("*.csv")}
missing = required - actual
print("成果文件：", sorted(actual))
assert not missing, f"缺少成果文件：{sorted(missing)}"
print("第9天Notebook检查通过")

成果文件： ['baseline_metrics.csv', 'feature_schema.csv', 'model_matrix_preview.csv', 'split_summary.csv']
第9天Notebook检查通过
